In [2]:
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
import os

from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.agents.run_config import RunConfig
from google.genai import types

from utils.tools import check_warehouse_availability, reserve_warehouse_items


/home/arminas_work/projects/AI-Engineering_Bootcamp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### ADK Agent

In [3]:
model = LiteLlm(
    model="openai/gpt-4.1-mini",
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY")
)

In [5]:
warehouse_agent = Agent(
    name="warehouse_manager_agent",
    model=model,
    tools=[check_warehouse_availability, reserve_warehouse_items],
    description="Agent is able to reserve items from the warehouses or check the availability of the items in warehouses",
    instruction="""
You are a part of the shopping assistant that can manage available inventory in the warehouses.

You will be given a conversation history and a list of tools, your task is to perform actions requested by the latest user query. Answer part of the query that you can answer with the available tools.

Instructions:
- You must always check the availability of the items in the warehouses before reserving them.
- Only reserve items in warehouses if entire order can be reserved or the user has confirmed that they want a partial reservation.
- If you cannot reserve any items, return an answer that the order cannot be reserved.
- If you can reserve some items, return an answer that the order can be partially reserved and include the details.
- If only partial quantity can be reserved in some warehouses, try to combinethe required quantity from different warehouses.
- Try to reserve items from the closest warehouse to the user first if users location is provided.
"""
)

### ADK Session

In [6]:
session_service = InMemorySessionService()

In [7]:
await session_service.create_session(
    app_name="warehouse_app",
    user_id="user_1",
    session_id="session_1"
)

Session(id='session_1', app_name='warehouse_app', user_id='user_1', state={}, events=[], last_update_time=1772106321.2019765)

In [10]:
runner = Runner(
    agent= warehouse_agent,
    session_service=session_service,
    app_name="warehouse_app"
)

In [13]:
item_id= "B0CC61QC4J"

message = types.Content(
    role = "user",
    parts = [types.Part(text=f"Hello, I want to check all warehouses that have this specific item- {item_id} available")]
)

In [14]:
result = runner.run(
    user_id="user_1",
    session_id="session_1",
    new_message= message,
    run_config=RunConfig(
        max_llm_calls=3
    )
)

In [15]:
for event in result:
    print("="*20)
    print(event)

model_version='gpt-4.1-mini-2025-04-14' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'items': [
            {<... 2 items at Max depth ...>},
          ]
        },
        id='call_I5EgeZdfabLHeMZwJKfUNKbH',
        name='check_warehouse_availability'
      )
    ),
  ],
  role='model'
) grounding_metadata=None partial=False turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  cached_content_token_count=0,
  candidates_token_count=32,
  prompt_token_count=516,
  total_token_count=548
) live_session_resumption_update=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None interaction_id=None invocation_id='e-aff74f76-53de-4b3e-a1fd-62102037983a' author='warehouse_manager_agent' actions=EventActions(skip_summarization=None

In [16]:
session_service = InMemorySessionService()

async def ask_warehouse_agent(query: str, session_id: str):

    existing_session = await session_service.get_session(
        app_name="warehouse_app",
        user_id="user_1",
        session_id=session_id,
    )

    if not existing_session:
        await session_service.create_session(
            app_name="warehouse_app",
            user_id="user_1",
            session_id=session_id,
        )

    runner = Runner(
        agent=warehouse_agent,
        app_name="warehouse_app",
        session_service=session_service,
    )

    content = types.Content(
        role="user",
        parts=[types.Part(text=query)],
    )

    final_text = ""

    for event in runner.run(
        user_id="user_1",
        session_id=session_id,
        new_message=content,
        run_config=RunConfig(
            max_llm_calls=3,
        ),
    ):
        if event.is_final_response():
            if event.content and event.content.parts:
                for part in event.content.parts:
                    final_text += part.text
            break

    return final_text


In [19]:
item_id= "B0CC61QC4J"

result = await ask_warehouse_agent(
    query=f"Hello, I want to check all warehouses that have this specific item- {item_id} available",
    session_id="1"
    )

In [21]:
print(result)

The item with product ID B0CC61QC4J is available in the following warehouses, each able to fulfill the request completely:

- Berlin Distribution Center, Berlin, Germany (72 units available)
- Lyon Regional Warehouse, Lyon, France (23 units available)
- Munich Logistics Hub, Munich, Germany (33 units available)
- Paris Central Depot, Paris, France (60 units available)
- Marseille Mediterranean Hub, Marseille, France (77 units available)
- Hamburg North Warehouse, Hamburg, Germany (88 units available)

If you want, I can help you reserve this item from any of these warehouses.


In [22]:
result_2 = await ask_warehouse_agent(
    query="Reserve 2 items in Lyon",
    session_id="1"
    )

In [23]:
print(result_2)

I have successfully reserved 2 units of the item with product ID B0CC61QC4J from the Lyon Regional Warehouse in Lyon, France. If you need any further assistance, please let me know.
